# Egatur · Gasto turístico en España

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolenki7/ETL_Egatur_INE_Esp/blob/main/Egatur_Power_BI_project.ipynb)

## La pregunta de negocio

España recibe turismo de muchos mercados y no todos dejan el mismo dinero ni llegan por la misma puerta. La pregunta no es *cuánto se gasta* sino **de dónde viene ese gasto, dónde se queda y por dónde entra** — porque cada una de esas respuestas apunta a una decisión distinta: a qué mercado promocionar, qué comunidad atender y qué infraestructura priorizar.

**Datos.** Cinco tablas de la encuesta **Egatur del INE**, ya extraídas y unificadas en `data/processed/egatur_master_long.csv` (2.881 filas, formato largo: `table_id`, `dimension_value`, `periodo`, `valor`).

| Tabla | Dimensión | Granularidad |
|:--|:--|:--|
| 10838 | País de residencia | mensual |
| 10839 | CCAA de destino | mensual |
| 10828 | Vía de acceso | mensual |
| 13938 | Partidas de gasto | mensual |
| 23995 | Motivo del viaje | **anual** |


## 1. Primero, la trampa: granularidad mixta

Antes de sumar nada hay que comprobar en qué unidades de tiempo viene cada fila. En este master **conviven periodos mensuales (`2026M07`) y anuales (`2025`)**: la tabla de motivo del viaje solo se publica por años. Si se suma la columna `valor` sin separarlas, el mismo gasto se cuenta dos veces y el resultado es basura silenciosa.

Por eso el análisis separa primero las dos granularidades y trabaja con **cuotas** (porcentajes del total), que además evitan el problema de las unidades: el INE publica estas series en miles de euros, pero el CSV no lo declara en ninguna columna, así que convertir a euros sería una suposición. Una cuota no lo es.


In [ ]:
import pathlib

import pandas as pd

try:
    from src.svg_chart import AZUL, ROJO, barras
except ImportError:  # en Colab el repo no está clonado
    import urllib.request
    url = ('https://raw.githubusercontent.com/Nicolenki7/'
           'ETL_Egatur_INE_Esp/main/src/svg_chart.py')
    exec(urllib.request.urlopen(url).read().decode())

d = pd.read_csv('data/processed/egatur_master_long.csv')
d['mensual'] = d['periodo'].astype(str).str.fullmatch(r'\d{4}M\d{2}')
d['fecha'] = pd.to_datetime(d['periodo_parsed'], format='%Y-%m', errors='coerce')
print(f'{len(d):,} filas · {d.table_id.nunique()} tablas')
print(f'mensuales: {int(d.mensual.sum()):,} · anuales: {int((~d.mensual).sum()):,}')
print()
print(d.groupby(['table_id', 'table_name']).agg(
    filas=('valor', 'size'),
    mensuales=('mensual', 'sum')).to_string())


## 2. Dónde se queda el gasto

Se toma una ventana de **12 meses** cerrada en el último mes disponible, para no comparar un año parcial con uno completo. Todos los cortes usan la misma ventana y la misma fórmula: suma del gasto del grupo entre suma total de la tabla.


In [ ]:
fin = d[d.mensual].fecha.max()
ini = fin - pd.DateOffset(months=11)
ventana = f'{ini:%Y-%m} → {fin:%Y-%m}'
print('ventana:', ventana)

def cuotas(table_id):
    g = d[(d.table_id == table_id) & d.mensual
          & d.dimension_value.ne('Total') & d.fecha.between(ini, fin)]
    s = g.groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
    return s / s.sum() * 100

IMG = pathlib.Path('docs/img')
ETIQUETA_CCAA = {'09 Cataluña': 'Cataluña', '05 Canarias': 'Canarias',
                 '04 Balears, Illes': 'Baleares', '01 Andalucía': 'Andalucía',
                 '13 Madrid, Comunidad de': 'Madrid',
                 '10 Comunitat Valenciana': 'C. Valenciana',
                 'Otras Comunidades Autónomas': 'Otras'}

ccaa = cuotas(10839)
print(ccaa.round(1).to_string())
barras(
    [(ETIQUETA_CCAA.get(i, i), v, AZUL) for i, v in ccaa.items()],
    'El gasto turístico está repartido, no concentrado',
    IMG / 'egatur-ccaa.svg',
    subtitulo=f'% del gasto por comunidad autónoma · {ventana}',
    formato='{:.1f}', sufijo='%')


In [ ]:
pais = cuotas(10838)
print(pais.round(1).to_string())
barras(
    [(i, v, ROJO if v > 50 else AZUL) for i, v in pais.items()],
    'La mitad del gasto no es de los cuatro mercados clásicos',
    IMG / 'egatur-pais.svg',
    subtitulo=f'% del gasto por país de residencia · {ventana}',
    formato='{:.1f}', sufijo='%')

via = cuotas(10828)
print()
print(via.round(1).to_string())
barras(
    [(i, v, ROJO if v > 50 else AZUL) for i, v in via.items()],
    'Nueve de cada diez euros entran por avión',
    IMG / 'egatur-via.svg',
    subtitulo=f'% del gasto por vía de acceso · {ventana}',
    formato='{:.1f}', sufijo='%')


## 3. Tendencia y motivo

La tendencia se mide **comparando los mismos meses** (enero a julio de 2025 contra enero a julio de 2026): comparar un año en curso contra un año completo daría una caída falsa. El motivo del viaje, al ser anual, se lee aparte y no se suma con el resto.


In [ ]:
t = d[(d.table_id == 10838) & (d.dimension_value == 'Total') & d.mensual]
en25 = t[(t.fecha.dt.year == 2025) & (t.fecha.dt.month <= 7)]['valor'].sum()
en26 = t[t.fecha.dt.year == 2026]['valor'].sum()
print(f'ene-jul 2025 = {en25:,.0f} · ene-jul 2026 = {en26:,.0f} · variación {(en26/en25-1)*100:+.1f}%')

mot = d[(d.table_id == 23995) & (d.periodo.astype(str) == '2025')
        & d.dimension_value.ne('Total')]
mot = mot.groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
print()
print('motivo del viaje 2025 (% del gasto):')
print((mot / mot.sum() * 100).round(1).head(6).to_string())

pathlib.Path('outputs').mkdir(exist_ok=True)
pd.DataFrame({'ccaa': ccaa, 'pais': pais}).to_csv('outputs/share_by_geo.csv')


## Qué dicen los números

- **El gasto está repartido entre comunidades:** Cataluña 18,8 %, Canarias 17,3 %, Baleares 15,4 %, Andalucía 15,0 %, Madrid 13,5 %, Comunidad Valenciana 12,0 %. Ninguna llega al 20 %: no hay una comunidad que domine, y siete destinos se reparten el 92 %.
- **El mercado está mucho más diversificado de lo que sugiere el relato habitual:** Reino Unido 17,2 %, Alemania 11,5 %, Francia 8,4 %, Nórdicos 5,7 %, Italia 4,2 %, y un **53,0 % en *Resto del Mundo***. Más de la mitad del gasto viene de mercados que la tabla agrupa en una sola etiqueta.
- **La puerta de entrada es una sola:** el **89,6 % del gasto entra por aeropuerto**, frente al 8,2 % por carretera, 1,7 % por puerto y 0,5 % por tren. Cualquier incidencia en el transporte aéreo afecta a casi todo el negocio.
- **La tendencia es positiva:** enero-julio de 2026 está un **+7,8 %** por encima del mismo tramo de 2025.
- **El motivo manda el ocio:** 44,7 % ocio/recreo/vacaciones, 19,3 % sol y playa, 12,4 % otro ocio, 12,2 % cultural. El negocio es un 88,6 % ocio; lo corporativo (2,8 %) y las visitas a familiares (2,6 %) son marginales en gasto.

### Limitaciones

- **Las tablas no son sumables entre sí**: son indicadores distintos de la misma encuesta. Sumar el gasto por país con el gasto por CCAA duplicaría la cifra.
- **La tabla 23995 es anual y el resto mensual.** Cualquier modelo (Power BI incluido) que cruce ambas sin filtrar por granularidad da resultados inflados.
- **El CSV no declara unidades.** El INE publica estas series en miles de euros, pero al no venir en columna no se puede verificar desde el fichero: por eso el análisis trabaja con cuotas y no con importes.
- **Las fechas del master llegan hasta 2026-07**, lo que es el último periodo publicado en la extracción; no hay garantía de que sea el último dato disponible en el INE hoy.
- **`Resto del Mundo` es una caja negra** del 53 %: sin desagregar no se puede saber qué mercados crecen dentro de ella.

### Siguiente paso

Desagregar *Resto del Mundo* y cruzar país con comunidad (qué mercado va a qué destino) para pasar de *dónde se gasta* a *qué campaña mover*. Ese cruce no está en estas cinco tablas: requeriría una tabla adicional del INE.
